# Module 3.4: Belief Revision

Facts change. People move cities, switch hotel chains, update dietary preferences.
What happens when an agent's memory says *"Sarah lives in New York"* but she just
told it she moved to Paris?

> **The question**: How should an agent handle facts that change over time?

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, os, sniffio, certifi
sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider)
from azure.ai.projects.aio import AIProjectClient as AsyncAIProjectClient
from openai import AsyncAzureOpenAI
from agent_framework import Agent, AgentSession, tool
from lifecycle_utils import GraphBeliefStore
from shared.travel_agent import (
    create_client, SYSTEM_PROMPT, search_flights, search_hotels, get_travel_policy)

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Foundry client ready")

## Setup: Connect to Neo4j

Same semantic memory graph from Module 2.3 / 3.3. Preferences live as nodes with
vector embeddings. The `GraphBeliefStore` in
[lifecycle_utils.py](lifecycle_utils.py) adds bi-temporal properties
(`valid_from`, `valid_to`, `superseded_by`) directly on those nodes.

In [ ]:
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.llm.adapters.openai import OpenAIProvider, OpenAIEmbeddingProvider
from pydantic import SecretStr

project_client = AsyncAIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"], credential=AsyncCliCredential())
llm = OpenAIProvider(model=os.environ.get("FOUNDRY_MODEL", "gpt-4o"))
llm._client = project_client.get_openai_client()

embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(), "https://cognitiveservices.azure.com/.default")
embedder = OpenAIEmbeddingProvider(
    model=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002"))
embedder._client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token, api_version="2024-02-01")
print("LLM + embedding providers ready.")

## The Problem: Two Naive Approaches, Both Wrong

| Approach | Mechanism | Failure |
|----------|-----------|---------|
| **Overwrite** (latest wins) | Delete old node, write new | History lost. "Where did Sarah live in January?" → unanswerable |
| **Append-only** | Add new node, keep old | Contradictions. Recall returns "NYC" AND "Paris" — agent can't tell which is current |

The root cause: neither tracks **when a fact was true in the real world**.

Cosine similarity cannot distinguish a contradicted fact from a duplicate —
RAG serves superseded values 15–40% of the time *(arXiv:2606.26511)*.

## The Solution: SCD Type 2 (Supersession)

Borrowed from data warehousing: every version of a fact is a separate row
with `valid_from` / `valid_to` timestamps. When a new value arrives, the old
row is *retired* (its `valid_to` is set) — never deleted.

```mermaid
stateDiagram-v2
    [*] --> Current : store belief
    Current --> Superseded : new value for same category
    Superseded --> [*] : retained for audit
    Current --> Current : same value confirmed
```

This gives us:
- **Current query** → "Where does Sarah live?" → Paris ✅
- **Time-travel query** → "Where did she live in January?" → NYC ✅
- **Full audit trail** → every change, with timestamps and provenance

In [ ]:
settings = MemorySettings(
    neo4j={"uri": os.environ["NEO4J_URI"], "username": os.environ["NEO4J_USER"],
           "password": SecretStr(os.environ["NEO4J_PASSWORD"]),

           "database": os.environ.get("NEO4J_DATABASE", "neo4j")},
    llm=llm, embedding=embedder)
print("GraphBeliefStore ready (SCD Type 2)")
print(f"Connected to Neo4j: {os.environ['NEO4J_URI']}")

memory = MemoryClient(settings)
await memory.__aenter__()
beliefs = GraphBeliefStore(memory)
await beliefs.reset()


## The Revision Agent

The agent gets four memory tools — thin wrappers over `GraphBeliefStore`:

| Tool | Purpose |
|------|---------|
| `remember_belief` | Store a preference, auto-superseding any conflicting current belief |
| `recall_current_beliefs` | Retrieve only currently-valid beliefs (superseded ones filtered out) |
| `recall_beliefs_at_time` | Time-travel: what did we believe on a specific date? |
| `belief_history` | Full SCD Type 2 audit trail for a category |

The agent's instructions tell it to use `source_type='user_assertion'` for
explicit user statements and `'llm_inference'` for anything derived.

In [ ]:
@tool
async def remember_belief(category: str, preference: str,
                          source_type: str = "user_assertion") -> str:
    """Store a belief. Supersedes any conflicting current belief in the same category.
    source_type: 'user_assertion' for explicit statements, 'llm_inference' for derived."""
    r = await beliefs.store(category, preference, source_type)
    sup = f" (superseded: '{r['superseded']}')" if r["superseded"] else ""
    return f"Stored '{preference}' as {r['state']}{sup}."

@tool
async def recall_current_beliefs(query: str) -> str:
    """Recall currently-valid beliefs. Superseded beliefs are excluded."""
    return await beliefs.recall_current(query)

@tool
async def recall_beliefs_at_time(query: str, date: str) -> str:
    """Time-travel: recall beliefs valid on a specific date (YYYY-MM-DD)."""
    return await beliefs.recall_at_time(query, date)

print("Store + recall tools defined.")

In [ ]:
@tool
async def belief_history(category: str) -> str:
    """Full audit trail for a belief category (all versions with timestamps)."""
    rows = await beliefs.history(category)
    if not rows:
        return f"No history for '{category}'."

    lines = []print("History tool defined.")

    for r in rows:

        vt = r['valid_to'][:10] if r['valid_to'] else 'present'    return "\n".join(lines)
        lines.append(f"{r['preference']} ({r['valid_from'][:10]} → {vt})")

In [ ]:
assistant = Agent(
    client=client, name="TravelAssistant",
    instructions=SYSTEM_PROMPT + "\n\n" + (
        "You manage the user's long-term preferences with belief revision.\n"
        "- When the user states a preference, call remember_belief with the "
        "appropriate category (e.g. 'home_city', 'hotel_chain', 'diet').\n"
        "- Use source_type='user_assertion' for explicit statements.\n"
        "- Before recommendations, call recall_current_beliefs.\n"
        "- For historical questions ('where did I live in Jan?'), use "
        "recall_beliefs_at_time.\n"
        "- If the user asks how a preference evolved, use belief_history."),
    tools=[search_flights, search_hotels, get_travel_policy,
           remember_belief, recall_current_beliefs,
           recall_beliefs_at_time, belief_history])
print(f"Agent ready: {assistant.name}")

## Demo: Sarah Moves Cities

We run the exact scenario that breaks overwrite and append-only.
The agent stores preferences, handles a city change (supersession),
and answers both current and historical queries correctly.

In [ ]:
session = AgentSession()

r1 = await assistant.run(
    "I live in New York — SFO is not my airport, JFK is. "
    "And I always stay at Marriott when travelling.",
    session=session)
print("ASSISTANT:", r1.text, "\n")
for p in await beliefs.snapshot():
    print(f"  [{p['state']:11s}] {p['category']:15s} → {p['preference']}")

In [ ]:
r2 = await assistant.run(
    "Big news — I just moved to Paris for work! CDG is my new home airport.",
    session=session)
print("ASSISTANT:", r2.text, "\n")
for p in await beliefs.snapshot():
    status = "CURRENT" if p["valid_to"] is None else "SUPERSEDED"
    print(f"  [{status:10s}] {p['category']:15s} → {p['preference']}")

In [ ]:
r3 = await assistant.run(
    "Wait, I need to file an expense from January. "
    "Where was I living back then? Which airport would I have used?",
    session=session)
print("ASSISTANT:", r3.text)

## Edge Case: Visiting vs Moving

Not every location mention is a permanent change. The agent's instructions
and the `category` parameter handle this naturally:

- *"I just moved to Paris"* → `remember_belief(category='home_city', ...)` → supersedes NYC
- *"I'm visiting Tokyo next week"* → the agent should NOT call `remember_belief` for `home_city`

The agent reasons about whether a statement represents a permanent change.
Let's test it:

In [ ]:
r4 = await assistant.run(
    "I'm visiting Tokyo next week for a conference. Can you find me a hotel?",
    session=session)
print("ASSISTANT:", r4.text, "\n")

# Verify: home_city should still be Paris, not Tokyo
print("Current beliefs after 'visiting Tokyo':")
for p in await beliefs.snapshot():
    if p["valid_to"] is None:
        print(f"  [{p['category']:15s}] {p['preference']}")

## Audit Trail: Full Belief History

Every change is preserved in Neo4j with timestamps. The agent can retrieve
the full evolution of any belief category — useful for compliance, debugging,
and explaining past recommendations.

In [ ]:
r5 = await assistant.run(
    "Show me the full history of where I've lived.",
    session=session)
print("ASSISTANT:", r5.text, "\n")

# Read directly from Neo4j for verification
print("Neo4j audit trail (home_city):")
for r in await beliefs.history("home_city"):
    vt = r["valid_to"][:10] if r["valid_to"] else "present"
    print(f"  {r['preference']:15s} | {r['valid_from'][:10]} → {vt}")

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **SCD Type 2** | Every version is a separate node with `valid_from` / `valid_to` |
| **Supersession** | Same category, new value → old node retired, not deleted |
| **Time-travel** | `recall_beliefs_at_time` filters by `valid_from ≤ T < valid_to` |
| **Audit trail** | `belief_history` returns full evolution with provenance |
| **Composability** | Beliefs still have `state` from Module 3.3 (trust promotion) |

1. **Never overwrite** — retire the old value with a `valid_to` timestamp
2. **Never append blindly** — supersession prevents contradictions in recall
3. **Two time dimensions** — valid-time (real world) and transaction-time (when stored)
4. **The agent drives it** — it stores, supersedes, and queries through tools
5. **Same Neo4j graph** — no separate store; temporal properties on existing Preference nodes

## Next: Retention & Decay (Notebook 05)

The belief store grows over time. The next notebook implements bounded memory
with scoring-based retention — ensuring old, unused, or redundant beliefs are
gracefully evicted rather than accumulated indefinitely.